In [ ]:
#Qué hace: identifica empeoramiento radiológico a partir de informes durante la evolución.
#Clave: marcador observado, sujeto a sesgo de indicación y subjetividad.

In [1]:
# 0. setup

from google.cloud import bigquery
import pandas as pd
import numpy as np
import re

PROJECT_ID = "mimic-pruebas"
NOTE = "physionet-data.mimiciv_note"   # ya comprobado

client = bigquery.Client(project=PROJECT_ID)

WINDOWS_PATH = "05_ventanas_24h.parquet"
OUT_PATH     = "15_radiology_worsening_daily.parquet"

In [2]:
# 1. Ventanas (filtramos cohortes para no bajar 2.3M informes completos)

df_win = pd.read_parquet(WINDOWS_PATH)[
    ["subject_id","hadm_id","icu_stay_id","day_idx","window_start","window_end"]
].copy()

df_keys = df_win[["subject_id","hadm_id"]].drop_duplicates()

hadm_ids = df_keys["hadm_id"].dropna().astype(int).unique().tolist()
hadm_ids_sql = ",".join(map(str, hadm_ids))

In [3]:
# 2. Descargar SOLO radiología de tu cohorte

sql_rad = f"""
SELECT
  subject_id,
  hadm_id,
  charttime,
  text
FROM `{NOTE}.radiology`
WHERE hadm_id IN ({hadm_ids_sql})
  AND charttime IS NOT NULL
"""
df_rad = client.query(sql_rad).to_dataframe()
df_rad["charttime"] = pd.to_datetime(df_rad["charttime"])

E0000 00:00:1769606582.158829  953340 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


In [4]:
# 3. Descargar SOLO radiología de tu cohorte

sql_rad = f"""
SELECT
  subject_id,
  hadm_id,
  charttime,
  text
FROM `{NOTE}.radiology`
WHERE hadm_id IN ({hadm_ids_sql})
  AND charttime IS NOT NULL
"""
df_rad = client.query(sql_rad).to_dataframe()
df_rad["charttime"] = pd.to_datetime(df_rad["charttime"])

E0000 00:00:1769606590.540832  953340 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


In [5]:
# 4) NLP: términos + negaciones (más seguro que el borrador)
# 4.1 Términos de empeoramiento (inglés, como tus notas)

WORSENING_PATTERNS = [
    r"\bnew (?:focal )?(?:consolidation|infiltrate|opacity|opacit(?:y|ies))\b",
    r"\binterval (?:increase|development|worsening|progression)\b",
    r"\bworsen(?:ing|ed)?\b",
    r"\bprogress(?:ion|ive)\b",
    r"\bincreas(?:ed|ing) (?:bilateral )?(?:opacity|opacit(?:y|ies)|infiltrate|consolidation)\b",
    r"\bmore (?:prominent|extensive)\b",
    r"\bdevelop(?:ed|ing)\b.*\b(?:infiltrate|consolidation|opacity|opacit(?:y|ies))\b",
]

# 4.2 Negaciones (evitar falsos positivos)
NEGATION_PATTERNS = [
    r"\bno (?:new|acute) (?:infiltrate|consolidation|opacity|opacit(?:y|ies))\b",
    r"\bwithout (?:evidence of )?(?:worsening|progression)\b",
    r"\bno significant (?:change|interval change)\b",
    r"\bunchanged\b",
    r"\bstable\b",
    r"\bimprov(?:ed|ing|ement)\b",
    r"\bresolv(?:ed|ing|e)\b",
    r"\bdecreas(?:ed|ing)\b.*\b(?:opacity|opacit(?:y|ies)|infiltrate|consolidation)\b",
]

# 4.3 Función de clasificación simple y trazable
neg_re = re.compile("|".join(NEGATION_PATTERNS), flags=re.IGNORECASE)
wor_re = re.compile("|".join(WORSENING_PATTERNS), flags=re.IGNORECASE)

def rad_worsening_flag(text: str) -> int:
    if not isinstance(text, str) or text.strip() == "":
        return 0
    # si hay negación fuerte, lo marcamos como no-empeoramiento
    if neg_re.search(text):
        return 0
    return int(bool(wor_re.search(text)))

df_rad["rad_worsening_evt"] = df_rad["text"].apply(rad_worsening_flag)

In [6]:
# 5) Asignar informe a ventana diaria
df_m = df_win.merge(
    df_rad[["subject_id","hadm_id","charttime","rad_worsening_evt"]],
    on=["subject_id","hadm_id"],
    how="left"
)

mask_in_window = (
    (df_m["charttime"] >= df_m["window_start"]) &
    (df_m["charttime"] <  df_m["window_end"])
)
df_m = df_m[mask_in_window].copy()

In [7]:
# 6) Agregación diaria + completar ceros

df_day = (
    df_m.groupby(["subject_id","hadm_id","icu_stay_id","day_idx"], as_index=False)
        .agg(
            rad_worsening_flag=("rad_worsening_evt","max"),
            n_rad_reports=("rad_worsening_evt","size")
        )
)

df_out = df_win.merge(df_day, on=["subject_id","hadm_id","icu_stay_id","day_idx"], how="left")
df_out["rad_worsening_flag"] = df_out["rad_worsening_flag"].fillna(0).astype(int)
df_out["n_rad_reports"] = df_out["n_rad_reports"].fillna(0).astype(int)

df_out.to_parquet(OUT_PATH, index=False)
df_out.head()

,subject_id,hadm_id,icu_stay_id,day_idx,window_start,window_end,rad_worsening_flag,n_rad_reports
0,11239107,25883588,32367987,0,2113-01-16 20:00:00,2113-01-17 20:00:00,0,4
1,11239107,25883588,32367987,1,2113-01-17 20:00:00,2113-01-18 20:00:00,0,6
2,11239107,25883588,32367987,2,2113-01-18 20:00:00,2113-01-19 20:00:00,0,2
3,11239107,25883588,32367987,3,2113-01-19 20:00:00,2113-01-20 20:00:00,0,2
4,11239107,25883588,32367987,4,2113-01-20 20:00:00,2113-01-21 20:00:00,0,2


In [8]:
# 7) QC mínimo (obligatorio)

print("Radiology rows pulled:", df_rad.shape)
print("Pct radiology reports flagged worsening:", df_rad["rad_worsening_evt"].mean())

print("Pct days with any radiology report:", (df_out["n_rad_reports"] > 0).mean())
print("Pct days with radiology worsening:", df_out["rad_worsening_flag"].mean())
print(df_out.groupby("day_idx")["rad_worsening_flag"].mean().head(10))

# QC extra (muy recomendable): revisar 20 positivos y 20 negativos

pos = df_rad[df_rad["rad_worsening_evt"]==1].sample(20, random_state=1)
neg = df_rad[df_rad["rad_worsening_evt"]==0].sample(20, random_state=1)

display(pos[["charttime","text"]].head(5))
display(neg[["charttime","text"]].head(5))

Radiology rows pulled: (226633, 5)
Pct radiology reports flagged worsening: 0.05214156808584804
Pct days with any radiology report: 0.4565641556328
Pct days with radiology worsening: 0.03683940527679748
day_idx
0    0.064499
1    0.049038
2    0.046095
3    0.042177
4    0.039678
5    0.037839
6    0.033421
7    0.036930
8    0.032964
9    0.034613
Name: rad_worsening_flag, dtype: float64


,charttime,text
62459,2169-03-01 21:00:00,INDICATION: Patient is a ___ female with ence...
183981,2177-05-27 04:31:00,SINGLE FRONTAL VIEW OF THE CHEST.\n\nREASON FO...
119973,2155-11-25 13:18:00,EXAMINATION: LUMBAR PUNCTURE (W/ FLUORO) N8 R...
64442,2174-07-30 22:32:00,HISTORY: Increased respiratory rate.\n\nFINDI...
66119,2152-06-02 17:52:00,EXAMINATION: Chest radiograph\n\nINDICATION: ...


,charttime,text
34169,2131-05-18 05:00:00,REASON FOR EXAMINATION: Evaluation of the pat...
70250,2152-01-14 05:19:00,EXAMINATION: CHEST (PORTABLE AP)\n\nINDICATIO...
32095,2124-08-21 07:38:00,EXAMINATION: CHEST (PORTABLE AP)\n\nINDICATIO...
17074,2136-07-04 03:00:00,"CHEST RADIOGRAPH\n\nINDICATION: Hypotension, ..."
184988,2205-07-14 07:36:00,CHEST RADIOGRAPH\n\nINDICATION: Chronic heart...
